# Finetuning Small Models: A Hands-On Guide

This notebook walks through the full finetuning pipeline for small language models: data curation, LoRA finetuning, inference, and evaluation.

It uses the `ftguide` package in this repo, which has two code paths:
- **Unsloth path**: for NVIDIA GPUs. 4-bit loading, fused kernels, fast training.
- **Fallback path**: for Mac/CPU development. Uses plain `transformers` + `peft`. Training is slow, but everything imports and runs enough to understand the machinery.

If your end goal is writing custom kernels, pay attention to Part 8. The fallback path here is the 'boring baseline' that custom kernels aim to accelerate.


## Part 1: Setup

### Installation

Two main knobs: GPU vs CPU, and Unsloth vs fallback.

For a real finetuning run on an NVIDIA GPU:
- `uv pip install unsloth trl peft datasets transformers accelerate bitsandbytes`
- Unsloth's install page has CUDA-specific wheels; use those for your PyTorch/CUDA version.

For reading and light development on a Mac or CPU machine:
- `uv pip install transformers peft datasets trl` is enough.
- Unsloth will not import, and the code automatically falls back.

The `ftguide` modules wrap heavy imports (`unsloth`, `torch`, `trl`, `datasets`) in `try/except`, so the notebook can import cleanly regardless.


In [ ]:
import sys
sys.path.insert(0, '../src')

import logging
logging.basicConfig(level=logging.INFO)

from ftguide.config import FinetuneConfig
from ftguide.data import DataCurator
from ftguide.trainer import Finetuner
from ftguide.inference import InferenceRunner
from ftguide.eval import Evaluator

try:
    import torch
    print(f'PyTorch version: {torch.__version__}')
    print(f'CUDA available: {torch.cuda.is_available()}')
except ImportError:
    print('PyTorch not installed. Training/inference cells will be skipped.')


## Config: the single source of truth

`FinetuneConfig` is a dataclass that holds every knob: model name, sequence length, LoRA settings, training hyperparameters, and data curation thresholds.

Key fields:
- `base_model`: the model to finetune. Default is `unsloth/Llama-3.2-1B`, a 1B parameter model.
- `max_seq_length`: context window. 2048 is a safe default.
- `lora_r` / `lora_alpha`: rank and scaling of the LoRA adapters.
- `target_modules`: which weight matrices get adapters. Usually all attention and MLP projections.
- `load_in_4bit`: quantize the base model to 4 bits to save VRAM.
- `per_device_train_batch_size` + `gradient_accumulation_steps`: effective batch size = `batch * accumulation`.
- `learning_rate`: 2e-4 is a typical LoRA LR; full finetuning uses lower LRs.
- `max_steps` / `num_train_epochs`: stop condition. We use `max_steps` for the demo.
- `min_quality_score` / `dedup_threshold`: data curation thresholds.


In [ ]:
# Create a config pointed at the local sample dataset
config = FinetuneConfig(
    base_model='unsloth/Llama-3.2-1B',
    dataset_path='../data/sample_dataset.jsonl',
    output_dir='outputs',
    max_steps=10,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    max_examples=None,
)

print(config)


## Part 2: Data Curation

For small models, data quality matters more than model size. A 1B model on clean, task-specific data often beats a 7B model on sloppy data.

The curation pipeline is:
1. **Load** the dataset.
2. **Score** each example for quality.
3. **Filter** out low-quality or over-length examples.
4. **Deduplicate** exact and near-duplicate examples.
5. **Format** examples into the training text the model sees.

We will run each step explicitly, then run the full `curate()` pipeline.


In [ ]:
curator = DataCurator(config)
raw_dataset = curator.load()

print(f'Loaded {len(raw_dataset)} examples')
print(f'Columns: {raw_dataset.column_names}')

# Show a few examples
for i, ex in enumerate(raw_dataset):
    print(f'\n--- Example {i} ---')
    print(f'Instruction: {ex["instruction"][:100]}')
    out = ex['output']
    preview = f'Output: {out[:150]}...' if len(out) > 150 else f'Output: {out}'
    print(preview)
    if i >= 3:
        break


### Quality scoring

`DataCurator.quality_score()` is a fast heuristic that runs before tokenization. It checks:
- **Length**: very short or extremely long texts are penalized.
- **Word diversity**: low `unique_words / total_words` suggests repetitive or templated text.
- **Alphanumeric ratio**: excessive special characters or garbage symbols hurt the score.
- **ASCII ratio**: for English datasets, lots of non-ASCII can be noise.
- **Repetition**: any 5-gram repeating more than 10% of the time triggers a penalty.

The score ranges from 0.0 to 1.0. It is not perfect, but it catches obvious junk quickly.


In [ ]:
# Score a few hand-picked examples to see what the heuristic catches
examples_to_score = [0, 2, 4, 5, 7]

for idx in examples_to_score:
    ex = raw_dataset[idx]
    text = '\n'.join([ex.get('instruction', ''), ex.get('input', ''), ex.get('output', '')])
    score = curator.quality_score(text)
    print(f'Example {idx}: score = {score:.2f}')
    print(f"  instruction: {ex['instruction'][:80]}")
    out_preview = ex['output'][:80].replace('\n', ' ')
    print(f"  output: {out_preview}")
    print()


In [ ]:
# Run the filter step
filtered = curator.filter_low_quality(raw_dataset)

print(f'Before filter: {len(raw_dataset)}')
print(f'After filter:  {len(filtered)}')


### Deduplication

Duplicate data is surprisingly common: scraped datasets contain repeated prompts, paraphrased answers, and template outputs. The curator does two passes:

1. **Exact dedup**: normalize whitespace, lowercase, and hash with MD5. Anything with the same hash is dropped.
2. **Near-dedup**: compare 5-word shingles with Jaccard similarity. If `datasketch` is installed it uses MinHash LSH (fast); otherwise it falls back to an O(n^2) comparison.

The threshold is controlled by `config.dedup_threshold` (default 0.9). A value of 0.9 means two examples sharing 90% of their shingles are considered near-duplicates.


In [ ]:
deduped = curator.deduplicate(filtered)

print(f'Before dedup: {len(filtered)}')
print(f'After dedup:  {len(deduped)}')
print(f'Stats: {curator.stats}')


### Formatting for supervised fine-tuning

SFTTrainer expects each example to have a `text` field containing the full sequence the model will read. For alpaca-style data we use the standard template:

```
Below is an instruction that describes a task, paired with an input...

### Instruction:
...
### Input:
...
### Response:
...
```

The model is trained to predict every token in this string. (Unsloth's `train_on_responses_only` can restrict loss to the response part, but that is off by default in this notebook.)


In [ ]:
formatted = curator.format_instructions(deduped)

print('Formatted example 0:')
print(formatted[0]['text'][:600])


In [ ]:
# Run the full curation pipeline in one shot
curator_full = DataCurator(config)
dataset = curator_full.load()
curated = curator_full.curate(dataset)

print(f'\nFinal curated dataset: {len(curated)} examples')


### Reading the report

The `report()` method prints length and token-count percentiles. Why care?
- **Context window**: if your 75th percentile is near `max_seq_length`, you will truncate a lot of examples.
- **Padding waste**: batches pad to the longest sequence. If lengths vary wildly, you waste compute on padding tokens.
- **Memory**: longer sequences use quadratically more attention memory.

If the report shows a huge spread, consider filtering by length or packing shorter examples together.


## Part 2.5: How to Know Your Data Is Actually Good

The curation pipeline above cleans your data. But clean does not mean good. A dataset can be perfectly deduplicated, well-formatted, and pass every quality filter - and still teach the model the wrong thing.

Here's how to actually assess data quality before spending GPU hours on training.


### Read 50-100 examples by hand

This is the single most important step and the one nobody does. You're looking for:
- Wrong answers presented as correct
- Formatting inconsistency (different prompt styles mixed together)
- Template artifacts (HTML tags, markdown remnants, boilerplate)
- Truncated or incomplete outputs
- Answers that don't match the instruction

If you can't explain what the model should learn by reading 20 random examples, the data is too noisy.


In [ ]:
# Print 5 random examples for manual inspection
import random
random.seed(42)
indices = random.sample(range(len(ds)), min(5, len(ds)))
for idx in indices:
    ex = ds[idx]
    print(f"--- Example {idx} ---")
    print(f"Instruction: {ex.get('instruction', 'N/A')[:200]}")
    print(f"Input: {ex.get('input', 'N/A')[:200]}")
    print(f"Output: {ex.get('output', 'N/A')[:300]}")
    print()


### Look at distributions, not just averages

Length histograms reveal problems averages hide. Bimodal distributions usually mean you're mixing two different task types. A long tail of very long examples eats your context window and wastes compute on padding.


In [ ]:
# Length distribution of the curated dataset
import matplotlib
matplotlib.use('Agg')  # works without display
import matplotlib.pyplot as plt

# Use the formatted text if available, else reconstruct
texts = [ex.get('text', ex.get('output', '')) for ex in ds]
lengths = [len(t) for t in texts]
approx_tokens = [l // 4 for l in lengths]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

ax1.hist(lengths, bins=30, edgecolor='black', alpha=0.7)
ax1.set_xlabel('Character length')
ax1.set_ylabel('Count')
ax1.set_title('Char length distribution')
ax1.axvline(sum(lengths)/len(lengths), color='red', linestyle='--', label=f'mean={sum(lengths)//len(lengths)}')
ax1.legend()

ax2.hist(approx_tokens, bins=30, edgecolor='black', alpha=0.7, color='orange')
ax2.set_xlabel('Approx tokens (chars//4)')
ax2.set_ylabel('Count')
ax2.set_title('Token length distribution')
ax2.axvline(config.max_seq_length, color='red', linestyle='--', label=f'max_seq_len={config.max_seq_length}')
ax2.legend()

plt.tight_layout()
plt.savefig('data_length_dist.png', dpi=100)
plt.show()
print("Saved length distribution plot.")


### N-gram repetition across the whole dataset

Near-duplicates are caught by dedup, but broader n-gram repetition is different. If the same 5-gram appears in 50% of examples, the model will memorize that phrase regardless of dedup. This is common in template-generated datasets.


In [ ]:
# Check top repeated 5-grams across the dataset
from collections import Counter

all_5grams = Counter()
for ex in ds:
    text = ex.get('text', ex.get('output', ''))
    words = text.lower().split()
    for i in range(len(words) - 4):
        gram = ' '.join(words[i:i+5])
        all_5grams[gram] += 1

total = len(all_5grams)
top_10 = all_5grams.most_common(10)
print(f"Total unique 5-grams: {total}")
print(f"Total 5-gram instances: {sum(all_5grams.values())}")
print(f"\nTop 10 most repeated 5-grams:")
for gram, count in top_10:
    pct = 100 * count / len(ds)
    print(f"  [{count:3d}x in {pct:.0f}% of examples] {gram[:80]}")


### Perplexity: the ultimate signal

Run your dataset through the base model and measure perplexity. This tells you how 'surprising' your data is to the model:
- Very high perplexity = out of distribution. Wrong language, wrong format, or garbage. The model can't learn from it.
- Very low perplexity = the model already knows this. You're wasting compute.
- The sweet spot is moderately surprising data - new knowledge or skills the model is close to but not quite there.

This requires a loaded model, so it goes after the model loading section. For now, keep it in mind.


## Part 3: Loading and Preparing the Model

`Finetuner.load_model()` picks the best available loader:
- **Unsloth path**: `FastLanguageModel.from_pretrained(..., load_in_4bit=True)` loads the base model into 4-bit NormalFloat format. This keeps the 1B model under ~1 GB of VRAM.
- **Fallback path**: `transformers.AutoModelForCausalLM.from_pretrained()` loads in full precision. This works on CPU but uses ~2-4 GB of RAM and is slow.

Custom kernels usually replace the fallback's naive PyTorch operations with fused, quantized, or memory-optimized equivalents.


In [ ]:
# Load the model. On a Mac without unsloth this may take a while and use several GB of RAM.
model, tokenizer = None, None

try:
    finetuner = Finetuner(config)
    model, tokenizer = finetuner.load_model()
    print(f'Model loaded: {type(model).__name__}')
    print(f'Tokenizer vocab size: {len(tokenizer)}')
except Exception as e:
    print(f'Model loading failed: {e}')
    print('This is expected on CPU-only machines or when the model is not downloaded.')


## LoRA: parameter-efficient finetuning

LoRA (Low-Rank Adaptation) freezes the base model weights and injects small trainable adapter matrices into selected layers. Instead of updating a full weight matrix `W`, it learns `W + BA` where `B` and `A` are skinny matrices of rank `r`.

Why it works:
- The intrinsic dimension of the update needed for most tasks is much smaller than the full weight dimension.
- Training only `BA` uses far less memory and compute.
- You can swap adapters in and out without reloading the base model.

The target modules here are:
- `q_proj`, `k_proj`, `v_proj`, `o_proj`: attention projections.
- `gate_proj`, `up_proj`, `down_proj`: MLP projections in Llama-style models.

`lora_alpha` scales the adapter output. A common rule is `alpha = r` or `alpha = 2 * r`.


In [ ]:
if model is not None:
    model = finetuner.attach_lora(model)

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    print(f'Trainable params: {trainable:,} / {total:,} ({100 * trainable / total:.4f}%)')
else:
    print('Skipping LoRA attach because the model did not load.')


Only a tiny fraction of parameters are trainable. That is the point: most of the model's knowledge stays frozen, and the adapters learn the task-specific steering. The base weights remain in 4-bit (Unsloth path) or full precision (fallback), but only the adapters get gradients.


## Part 4: Training

`Finetuner.setup_trainer()` builds a TRL `SFTTrainer` with `SFTConfig`.

Key hyperparameters:
- `learning_rate=2e-4`: LoRA can use a higher LR than full finetuning because the update space is small.
- `per_device_train_batch_size=1` + `gradient_accumulation_steps=2`: effective batch size = 2. Increase accumulation if VRAM is tight.
- `warmup_steps=5`: slowly ramps the LR to avoid early instability.
- `max_steps=10`: short demo run. Real runs use hundreds or thousands of steps.
- `weight_decay=0.01`: regularizes the adapters.
- `lr_scheduler_type='linear'`: decays LR to zero over the run.
- `optim='adamw_8bit'`: 8-bit AdamW saves optimizer state memory. Falls back automatically if not available.


In [ ]:
if model is not None:
    trainer = finetuner.setup_trainer(model, tokenizer, curated)
    print(f'Trainer ready: {type(trainer).__name__}')
else:
    print('Skipping trainer setup because the model did not load.')
    trainer = None


In [ ]:
if trainer is not None:
    try:
        metrics = finetuner.train(trainer)
        print('Training finished. Metrics:')
        for k, v in metrics.items():
            print(f'  {k}: {v}')
    except Exception as e:
        print(f'Training failed: {e}')
        print('This is normal on CPU or when the model is too large for available RAM.')
else:
    print('No trainer available to run.')


### Reading training metrics

The main metric is `train_loss`. What to watch for:
- **Loss should decrease**: if it plateaus immediately, your LR may be too low or your data too small.
- **Loss should not spike**: a sudden jump means the LR is too high or a bad batch got through.
- **Overfitting**: if eval loss rises while train loss falls, you are memorizing the training set. Use more data, dropout, or shorter training.

On a 10-example dataset with 10 steps, do not expect a good model. This is just plumbing.


## Part 4.5: Preventing Overfitting

Overfitting with LoRA is easier than people think. A 1B model with r=16 adapters can memorize a few hundred examples in under 100 steps. Here's how to detect and prevent it.


### Always hold out 10-20% of your data

Never train on your full dataset. Split it: 80% train, 20% eval. Monitor eval loss alongside train loss. The moment eval loss starts rising while train loss keeps dropping, you're overfitting. Stop and use the checkpoint from before eval started rising.

```
step  train_loss   eval_loss
 10    2.4          2.5        both dropping, good
 20    1.8          1.9        both dropping, good
 30    1.3          1.4        both dropping, good
 40    0.9          1.0        both dropping, good
 50    0.6          0.8        gap widening, watch
 60    0.4          0.9        eval rising, STOP
 70    0.2          1.2        overfit, too late
```

The checkpoint from step 40-50 is your model. Not the last one.


In [ ]:
# Split the curated dataset into train and eval
# Use a fixed seed for reproducibility
from datasets import Dataset

split = ds.train_test_split(test_size=0.2, seed=42)
train_ds = split['train']
eval_ds = split['eval']
print(f"Train: {len(train_ds)} examples")
print(f"Eval:  {len(eval_ds)} examples")
print("Never let eval leak into train. This is your canary.")


### LoRA rank and dropout are your main levers

Lower rank = less capacity = harder to overfit. If you're overfitting:
1. Drop `lora_r` from 16 to 8. This is the single biggest lever.
2. Add `lora_dropout=0.05` to 0.1. The default is 0.0, which is aggressive.
3. Reduce learning rate from 2e-4 to 1e-4.
4. Fewer epochs. 1-3 is enough for most tasks with LoRA.
5. Bump `weight_decay` from 0.01 to 0.05.

Try them in that order. Don't change everything at once or you won't know what helped.


In [ ]:
# Anti-overfit config adjustments - apply if you see eval loss rising
anti_overfit_config = FinetuneConfig(
    lora_r=8,              # halved from 16
    lora_dropout=0.05,     # added dropout
    learning_rate=1e-4,    # halved from 2e-4
    weight_decay=0.05,     # 5x from 0.01
    max_steps=60,          # keep short
)
print("Anti-overfit config:")
print(f"  lora_r:       {anti_overfit_config.lora_r}")
print(f"  lora_dropout: {anti_overfit_config.lora_dropout}")
print(f"  lr:           {anti_overfit_config.learning_rate}")
print(f"  weight_decay: {anti_overfit_config.weight_decay}")
print("Apply these only if eval loss is rising. Otherwise stick with defaults.")


### Generate during training, not just after

Loss is a compressed signal. Actually generate 3-5 samples every 50 steps and read them. Overfitting shows up as:
- Outputs that are near-copies of training examples (memorization)
- Repetitive phrasing the model didn't have before
- Ignoring the instruction and regurgitating training data
- Losing the base model's general capability

Trust your eyes over the loss number. A model with low loss that generates garbage on new prompts is overfit, regardless of what the curve says.

### Catastrophic forgetting vs overfitting

These are different problems that look similar:
- Overfitting: model memorizes training data, loses generalization to new examples of the same task.
- Catastrophic forgetting: model forgets base model capabilities entirely. It gets great at your task but can't answer basic questions anymore.

For small models, forgetting is often worse than overfitting because the base model was already not that capable. The fix: lower learning rate, fewer steps, and consider mixing in some general data alongside your task data.


## Part 5: Saving and Exporting

There are three common save modes:
1. **LoRA adapters only**: small files (~tens of MB). Fast to save/load. Requires the base model to reconstruct the full weights.
2. **Merged 16-bit**: base + adapters fused into a single model. Easier to deploy but larger.
3. **GGUF**: quantized binary for `llama.cpp` / local inference tools. Not shown here because it requires Unsloth and a specific build.

For experimentation, save adapters. For deployment, merge and optionally quantize.


In [ ]:
if model is not None:
    finetuner.save(model, tokenizer, path='outputs/lora_model')
    print('Saved LoRA adapters to outputs/lora_model')
else:
    print('Skipping save because the model did not load.')


In [ ]:
if model is not None:
    try:
        finetuner.save_merged(model, tokenizer, path='outputs/merged_model')
        print('Saved merged 16-bit model to outputs/merged_model')
    except Exception as e:
        print(f'Merged save failed: {e}')
        print('Merged save requires Unsloth. Adapters were already saved separately.')
else:
    print('Skipping merged save.')


## Part 6: Inference

For inference, Unsloth calls `FastLanguageModel.for_inference(model)` to swap in optimized kernels and disable dropout. The fallback path just calls `model.eval()`.

Generation hyperparameters:
- `max_new_tokens`: hard cap on output length.
- `temperature`: 0 = greedy, >0 = random sampling. 0.7 is a common default.
- `do_sample`: whether to sample or take the argmax.
- `repetition_penalty`: penalizes repeated tokens.

The `InferenceRunner` class also handles chat templates for instruct-tuned models.


In [ ]:
runner = InferenceRunner(config)

if model is not None:
    # Reuse the model already in memory from training
    runner.model = model
    runner.tokenizer = tokenizer
    print('Reusing trained model for inference')
else:
    try:
        runner.load('outputs/lora_model')
        print('Loaded LoRA adapters for inference')
    except Exception as e:
        print(f'Inference loading failed: {e}')


In [ ]:
if runner.model is not None:
    prompts = [
        'Explain recursion in programming.',
        'Write a haiku about machine learning.',
    ]

    for prompt in prompts:
        print(f'\nPrompt: {prompt}')
        response = runner.generate(prompt, max_new_tokens=64, temperature=0.7)
        print(f'Response: {response}')

    # Chat interface
    messages = [
        {'role': 'system', 'content': 'You are a helpful coding assistant.'},
        {'role': 'user', 'content': 'What is the longest common subsequence problem?'},
    ]
    print('\nChat response:')
    print(runner.chat(messages, max_new_tokens=64, temperature=0.7))
else:
    print('No model loaded; skipping generation.')


### Inference tips

- Start with `temperature=0.7` and `max_new_tokens=128`. Adjust based on output length and hallucinations.
- Use `do_sample=False` for deterministic evals.
- Set `repetition_penalty=1.1` or `1.2` if the model repeats itself.
- For instruct models, always use `apply_chat_template` (the runner does this automatically in `chat()`).
- Batch generation is just a loop here. For real throughput, use continuous batching or a dedicated serving framework.


## Part 7: Evaluation

Evaluation is where you find out if the model actually learned anything or just memorized the data. `Evaluator` runs greedy generation on a few samples and computes simple metrics:

- **Exact match rate**: fraction of outputs that match the reference exactly. Rarely useful for open-ended generation.
- **ROUGE-1**: unigram overlap (precision/recall/F1).
- **ROUGE-L**: longest common subsequence based F1.
- **BLEU-1**: unigram precision with brevity penalty.

These string-overlap metrics are limited. They do not capture factual correctness or code execution. For serious benchmarking use `lm-eval-harness`, `inspect`, or task-specific evals.


In [ ]:
if runner.model is not None:
    evaluator = Evaluator(runner.model, runner.tokenizer, config)
    try:
        eval_results = evaluator.evaluate(
            curated,
            num_samples=min(5, len(curated)),
            max_new_tokens=64,
        )
        print('Raw metrics:')
        print(eval_results['metrics'])
    except Exception as e:
        print(f'Evaluation failed: {e}')
else:
    print('No model loaded; skipping evaluation.')


In [ ]:
if 'eval_results' in dir() and eval_results is not None:
    Evaluator.print_results(eval_results)
else:
    print('No evaluation results to print.')


## Part 8: Next Steps - Toward Custom Kernels

This is the part that matters if you want to write your own training/inference kernels.

### How Unsloth makes this fast

Unsloth replaces parts of the standard Hugging Face forward/backward pass with hand-written Triton kernels. The main wins come from:
- **Fused operations**: combining matmuls, activations, and normalizations into one kernel reduces memory round-trips.
- **4-bit dequantization on the fly**: weights stay compressed in GPU memory and are dequantized inside the kernel.
- **Optimized attention**: flash-attention-style memory access patterns, avoiding materialized attention matrices.
- **Custom backward pass**: re-computing some activations instead of storing them, trading compute for memory.

### What you need to learn

1. **PyTorch autograd**: understand how `backward()` builds the computation graph and how `torch.autograd.Function` lets you define custom forward/backward.
2. **GPU memory hierarchy**: global memory, shared memory, registers, and why coalesced access matters.
3. **Triton**: OpenAI Triton is the easiest way to write fast GPU kernels in Python-like syntax. Start with matrix multiplication and reduction tutorials.
4. **Transformer internals**: re-derive the forward and backward pass for attention and MLP layers. You cannot optimize what you do not understand.

### Suggested progression

1. Profile the fallback path with PyTorch profiler. Find the hotspots.
2. Re-implement a single operation (e.g., a LoRA forward) as a custom `torch.autograd.Function`.
3. Write the same operation in Triton and compare correctness and speed.
4. Replace the fallback layer in this codebase with your kernel and verify training still converges.

### Resources

- Triton tutorials (OpenAI Triton repo and Philippe Tillet's matmul tutorial).
- Unsloth source code on GitHub: read `unsloth/kernels/` and the model files.
- PyTorch internals notes (e.g., Edward Yang's blog posts on autograd).
- CUDA mode YouTube channel for kernel optimization intuition.

The fallback path in `ftguide/trainer.py` and `ftguide/inference.py` is your reference. Any custom kernel you write should produce numerically identical outputs and a speed or memory win against that baseline.


## Part 9: Summary

The full pipeline:
1. **Config**: one dataclass controls everything.
2. **Curate**: load -> quality filter -> dedup -> format. Data quality is the biggest lever.
3. **Load model**: Unsloth 4-bit on GPU, full-precision fallback on CPU.
4. **LoRA**: attach small adapters to attention and MLP layers.
5. **Train**: SFTTrainer with LoRA-friendly hyperparameters.
6. **Save**: adapters, merged 16-bit, or GGUF.
7. **Inference**: chat templates and sampling.
8. **Evaluate**: overlap metrics plus your own task-specific evals.

For small models, spend your effort on data first, then on efficient training second. The custom-kernel work in Part 8 pays off only after the data and pipeline are solid.
